# HGCTM Revision — Lithology-Source Sensitivity at Fixed \(K=7\)

This notebook performs the **same final M7b sparse HGCTM experiment** under three alternative lithology assignments:

1. **Macrostrat** — the original submitted lithology covariate.
2. **GLiM** — an independent full-polygon global lithology source.
3. **Combined fallback** — Macrostrat is retained when it provides a specific class; GLiM is used only when Macrostrat is `other` or `unknown`.

All three models use:

- the same 1,237 valid-coordinate deposits;
- the same 35 mineral-family count matrix;
- the same age categories;
- the same fixed \(K=7\);
- the same LDA warm start;
- the same M7b architecture;
- the same prior scales used by the submitted final model;
- the same optimization settings and random seeds.

The 98 records with placeholder coordinates `(0, 0)` are retained in the source database but excluded from this coordinate-derived lithology comparison.

## Scope

This notebook tests **sensitivity to the lithology source**. It does not test equal lithology/age priors; that is a separate reviewer-requested sensitivity analysis.

## Consistency with the submitted notebook

The model definition and settings were copied from the supplied original notebook:

- \(K=7\)
- `sigma_mu = 2.0`
- `sigma_litho = 0.3`
- `sigma_age = 0.15`
- Dirichlet–Multinomial likelihood
- per-family sparse gate \(	au_f \sim \mathrm{Beta}(1,3)\)
- learned age weight \(\omega_a \sim \mathrm{Beta}(2,2)\)
- LDA warm start
- 5,000 SVI steps
- \(	au\) annealing over 1,500 steps
- learning rate `0.005`

Original notebook SHA-256: `6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b`

## Required Kaggle inputs

Attach:

1. `Hgctm_stage0.zip` or the extracted Stage 0 dataset containing:
   - `X_family_copper.csv`
   - `covariates.csv`

2. `HGCTM_Phase1B_GLiM_Audit_Outputs.zip` or its extracted files containing:
   - `covariates_lithology_A_macrostrat_original.csv`
   - `covariates_lithology_B_glim_only.csv`
   - `covariates_lithology_C_macrostrat_glim_combined.csv`

No API requests are made in this notebook.

In [ ]:
# Install only packages that are missing.

import importlib.util
import subprocess
import sys

required = {
    "pyro": "pyro-ppl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cosine
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import StratifiedShuffleSplit

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_lithology_source_sensitivity"
RUNS_DIR = OUT / "runs"
FIG_DIR = OUT / "figures"

for folder in [OUT, RUNS_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ---------------- FINAL EXPERIMENT SETTINGS ----------------

K = 7
MIN_MINERALS = 3
MIN_DEPOSITS_PER_FAMILY = 10
EXPECTED_MODEL_N = 1335
EXPECTED_EXCLUDED_ZERO_COORDINATES = 98
EXPECTED_COMMON_N = 1237
EXPECTED_F = 35

FULL_SEEDS = [42, 123, 7]
PRIMARY_SEED = 42

N_STEPS = 5000
ANNEAL_END = 1500
LR = 0.005
PRINT_EVERY = 500

# Optional common random holdout. This is a predictive sanity check,
# not the corrected continent-transfer experiment.
RUN_COMMON_HOLDOUT = True
HOLDOUT_FRACTION = 0.20
HOLDOUT_SEED = 20260618
HOLDOUT_MODEL_SEED = 42

# Set True only for a quick code check. Do not use quick-mode results
# in the revised manuscript.
QUICK_TEST = False

if QUICK_TEST:
    ACTIVE_SEEDS = [42]
    ACTIVE_STEPS = 800
    ACTIVE_ANNEAL_END = 250
    RUN_COMMON_HOLDOUT = False
else:
    ACTIVE_SEEDS = FULL_SEEDS
    ACTIVE_STEPS = N_STEPS
    ACTIVE_ANNEAL_END = ANNEAL_END

LITHOLOGY_LEVELS = [
    "carbonate",
    "evaporite",
    "felsic",
    "mafic",
    "metamorphic",
    "other",
    "siliciclastic",
    "unknown",
    "volcanic",
]

AGE_LEVELS = [
    "Cenozoic",
    "Mesozoic",
    "Paleozoic",
    "Precambrian",
    "Unknown",
]

SOURCE_SPECS = {
    "Macrostrat": "lithology_A_class",
    "GLiM": "lithology_B_class",
    "Combined": "lithology_C_class",
}

print("PyTorch:", torch.__version__)
print("Pyro:", pyro.__version__)
print("Mode:", "QUICK TEST" if QUICK_TEST else "FINAL")
print("Seeds:", ACTIVE_SEEDS)


## 1. Locate and load the frozen inputs

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding="utf-8")
    return target


def find_folder_with(required_names, label):
    required_names = set(required_names)
    roots = [Path("/kaggle/input"), WORK]

    # Already extracted folder.
    for root in roots:
        if not root.exists():
            continue
        for first_name in required_names:
            for path in root.rglob(first_name):
                folder = path.parent
                present = {
                    item.name for item in folder.iterdir()
                    if item.is_file()
                }
                if required_names.issubset(present):
                    return folder

    # ZIP archive.
    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name for name in archive.namelist()
                    }
                    if required_names.issubset(basenames):
                        safe_label = re.sub(
                            r"[^a-z0-9]+", "_", label.lower()
                        ).strip("_")
                        target = WORK / f"{safe_label}_extracted"
                        return extract_zip_once(zip_path, target)
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        f"Could not locate {label}. Required files: "
        f"{sorted(required_names)}"
    )


STAGE0 = find_folder_with(
    {
        "X_family_copper.csv",
        "covariates.csv",
    },
    "HGCTM Stage 0",
)

PHASE1B = find_folder_with(
    {
        "covariates_lithology_A_macrostrat_original.csv",
        "covariates_lithology_B_glim_only.csv",
        "covariates_lithology_C_macrostrat_glim_combined.csv",
    },
    "HGCTM Phase 1B outputs",
)

print("Stage 0:", STAGE0)
print("Phase 1B:", PHASE1B)


In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def canonicalize_index(frame, label):
    frame = frame.copy()
    frame.index = [canonical_id(value) for value in frame.index]
    frame.index.name = "Mindat_id"

    if frame.index.isna().any():
        raise ValueError(f"{label} contains missing Mindat IDs.")
    if frame.index.duplicated().any():
        duplicates = frame.index[frame.index.duplicated()].tolist()
        raise ValueError(
            f"{label} contains duplicated IDs after normalization: "
            f"{duplicates[:10]}"
        )
    return frame


def canonicalize_id_column(frame, label):
    frame = frame.copy()
    if "Mindat_id" not in frame.columns:
        raise KeyError(f"{label} has no Mindat_id column.")

    frame["Mindat_id"] = frame["Mindat_id"].map(canonical_id)

    if frame["Mindat_id"].isna().any():
        raise ValueError(f"{label} contains missing Mindat IDs.")
    if frame["Mindat_id"].duplicated().any():
        duplicates = frame.loc[
            frame["Mindat_id"].duplicated(keep=False),
            "Mindat_id",
        ].tolist()
        raise ValueError(
            f"{label} contains duplicated IDs after normalization: "
            f"{duplicates[:10]}"
        )
    return frame


X_raw = canonicalize_index(
    pd.read_csv(STAGE0 / "X_family_copper.csv", index_col=0),
    "X_family_copper",
)
cov_stage0 = canonicalize_index(
    pd.read_csv(STAGE0 / "covariates.csv", index_col=0),
    "covariates",
)

source_a = canonicalize_id_column(
    pd.read_csv(
        PHASE1B / "covariates_lithology_A_macrostrat_original.csv"
    ),
    "Macrostrat covariates",
)
source_b = canonicalize_id_column(
    pd.read_csv(
        PHASE1B / "covariates_lithology_B_glim_only.csv"
    ),
    "GLiM covariates",
)
source_c = canonicalize_id_column(
    pd.read_csv(
        PHASE1B / "covariates_lithology_C_macrostrat_glim_combined.csv"
    ),
    "Combined covariates",
)

print("X raw:", X_raw.shape)
print("Stage 0 covariates:", cov_stage0.shape)
print("Phase 1B A/B/C:", source_a.shape, source_b.shape, source_c.shape)


## 2. Reconstruct the original 1,335-deposit model matrix

The original workflow:

1. retained deposits with at least three mapped mineral-family counts;
2. retained mineral families occurring in at least ten deposits.

The family filter is calculated on the original 1,335-deposit modelling set **before** the valid-coordinate cohort is selected. This preserves the same 35-family vocabulary used in the submitted model.

In [ ]:
model_mask = X_raw.sum(axis=1) >= MIN_MINERALS
X_model_pre_family_filter = X_raw.loc[model_mask].copy()

family_presence = (X_model_pre_family_filter > 0).sum(axis=0)
family_keep = family_presence[
    family_presence >= MIN_DEPOSITS_PER_FAMILY
].index.tolist()

X_model = X_model_pre_family_filter[family_keep].copy()

# The Stage 0 covariate file already contains the 1,335 model rows.
model_ids = [
    mindat_id for mindat_id in X_model.index
    if mindat_id in cov_stage0.index
]
X_model = X_model.loc[model_ids].copy()
cov_model = cov_stage0.loc[model_ids].copy()

print("Original modelling matrix:", X_model.shape)
print("Families:", list(X_model.columns))

if X_model.shape[0] != EXPECTED_MODEL_N:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_N} model deposits, "
        f"found {X_model.shape[0]}."
    )
if X_model.shape[1] != EXPECTED_F:
    raise ValueError(
        f"Expected {EXPECTED_F} retained families, "
        f"found {X_model.shape[1]}."
    )


## 3. Build the common 1,237-deposit valid-coordinate cohort

A row is excluded from this lithology-source comparison when:

- latitude or longitude is missing;
- coordinates are outside the valid geographic range;
- both coordinates are exactly zero.

No lithology source is allowed to use a different row set.

In [ ]:
def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    try:
        lat = float(lat)
        lon = float(lon)
    except Exception:
        return False
    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return False
    if abs(lat) < 1e-12 and abs(lon) < 1e-12:
        return False
    return True


a_small = source_a[
    [
        "Mindat_id",
        "Latitude",
        "Longitude",
        "lithology_A_class",
        "lithology_A_status",
    ]
].copy()

b_small = source_b[
    [
        "Mindat_id",
        "lithology_B_class",
        "lithology_B_status",
        "lithology_B_confidence",
        "glim_level1_code",
        "glim_level1_description",
    ]
].copy()

c_small = source_c[
    [
        "Mindat_id",
        "lithology_C_class",
        "lithology_C_status",
        "lithology_C_source",
        "lithology_C_confidence",
        "macrostrat_class",
        "macrostrat_status",
        "glim_model_class",
        "glim_status",
        "agreement_status",
    ]
].copy()

lithology_table = (
    a_small
    .merge(b_small, on="Mindat_id", how="inner", validate="one_to_one")
    .merge(c_small, on="Mindat_id", how="inner", validate="one_to_one")
)

lithology_table["coordinate_valid"] = lithology_table.apply(
    lambda row: valid_coordinate(row["Latitude"], row["Longitude"]),
    axis=1,
)

excluded_coordinates = lithology_table.loc[
    ~lithology_table["coordinate_valid"]
].copy()

common_ids_set = set(
    lithology_table.loc[
        lithology_table["coordinate_valid"],
        "Mindat_id",
    ]
)
common_ids = [
    mindat_id for mindat_id in X_model.index
    if mindat_id in common_ids_set
]

X_common = X_model.loc[common_ids].copy()
cov_common = cov_model.loc[common_ids].copy()

lithology_common = (
    lithology_table
    .set_index("Mindat_id")
    .loc[common_ids]
    .copy()
)

for column in SOURCE_SPECS.values():
    lithology_common[column] = (
        lithology_common[column]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

print("Excluded invalid-coordinate records:", len(excluded_coordinates))
print("Common valid-coordinate cohort:", len(common_ids))
print("Common matrix:", X_common.shape)

if len(excluded_coordinates) != EXPECTED_EXCLUDED_ZERO_COORDINATES:
    raise ValueError(
        f"Expected {EXPECTED_EXCLUDED_ZERO_COORDINATES} excluded records, "
        f"found {len(excluded_coordinates)}."
    )
if len(common_ids) != EXPECTED_COMMON_N:
    raise ValueError(
        f"Expected common cohort N={EXPECTED_COMMON_N}, "
        f"found N={len(common_ids)}."
    )

excluded_coordinates.to_csv(
    OUT / "excluded_invalid_coordinate_records.csv",
    index=False,
)


## 4. Audit the three lithology versions on the identical cohort

In [ ]:
unknown_labels = {}

for source_name, column in SOURCE_SPECS.items():
    values = set(lithology_common[column].dropna().unique())
    unexpected = values - set(LITHOLOGY_LEVELS)
    unknown_labels[source_name] = sorted(unexpected)

if any(unknown_labels.values()):
    raise ValueError(
        "Unexpected lithology labels detected: "
        f"{unknown_labels}"
    )

cohort = cov_common[
    [
        "Deposit_type",
        "age_category",
        "Latitude",
        "Longitude",
        "age_midpoint_ma",
    ]
].copy()

for source_name, column in SOURCE_SPECS.items():
    cohort[column] = lithology_common[column]

cohort["Mindat_id"] = cohort.index
cohort = cohort[
    [
        "Mindat_id",
        "Deposit_type",
        "age_category",
        "Latitude",
        "Longitude",
        "age_midpoint_ma",
        *SOURCE_SPECS.values(),
    ]
]

distribution_rows = []
for source_name, column in SOURCE_SPECS.items():
    counts = (
        cohort[column]
        .value_counts()
        .reindex(LITHOLOGY_LEVELS, fill_value=0)
    )
    for lithology_class, count in counts.items():
        distribution_rows.append({
            "source": source_name,
            "lithology_class": lithology_class,
            "count": int(count),
            "percent": 100 * count / len(cohort),
        })

source_distributions = pd.DataFrame(distribution_rows)

print("Lithology distributions on the common cohort:")
print(
    source_distributions.pivot(
        index="lithology_class",
        columns="source",
        values="count",
    ).fillna(0).astype(int).to_string()
)

cohort.to_csv(OUT / "common_cohort_covariates.csv", index=False)
X_common.to_csv(OUT / "common_cohort_X_family_copper.csv")
source_distributions.to_csv(
    OUT / "lithology_source_class_distributions.csv",
    index=False,
)


## 5. Fixed category encodings

A fixed union of nine lithology levels is used for all three models. This keeps the lithology parameter dimensions identical even when a source has no observations in one category.

The order matches the alphabetical encoding used by the original `LabelEncoder`.

In [ ]:
lithology_to_index = {
    label: index for index, label in enumerate(LITHOLOGY_LEVELS)
}
age_to_index = {
    label: index for index, label in enumerate(AGE_LEVELS)
}

age_values = (
    cohort["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)
unexpected_age = set(age_values.unique()) - set(AGE_LEVELS)
if unexpected_age:
    raise ValueError(f"Unexpected age labels: {sorted(unexpected_age)}")

age_idx_np = age_values.map(age_to_index).to_numpy(dtype=np.int64)

source_lithology_idx = {}
for source_name, column in SOURCE_SPECS.items():
    source_lithology_idx[source_name] = (
        cohort[column]
        .map(lithology_to_index)
        .to_numpy(dtype=np.int64)
    )

X_np = X_common.to_numpy(dtype=np.float32)
X_tensor = torch.tensor(X_np, dtype=torch.float32)
age_tensor = torch.tensor(age_idx_np, dtype=torch.long)

source_lithology_tensors = {
    source_name: torch.tensor(indices, dtype=torch.long)
    for source_name, indices in source_lithology_idx.items()
}

N, F = X_np.shape
L = len(LITHOLOGY_LEVELS)
A = len(AGE_LEVELS)

print(f"N={N}, F={F}, K={K}, L={L}, A={A}")
print("Lithology levels:", LITHOLOGY_LEVELS)
print("Age levels:", AGE_LEVELS)


## 6. Common LDA warm start

The LDA baseline is fitted once on the common mineral-family matrix. The same `phi_m1` is used to warm-start all Macrostrat, GLiM, and combined M7b fits.

In [ ]:
lda_m1 = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)

lda_m1.fit(X_np.astype(np.float64))

phi_m1 = (
    lda_m1.components_
    / lda_m1.components_.sum(axis=1, keepdims=True)
)
theta_m1 = lda_m1.transform(X_np.astype(np.float64))

np.save(OUT / "phi_m1_common_cohort.npy", phi_m1)
np.save(OUT / "theta_m1_common_cohort.npy", theta_m1)
pd.DataFrame(
    phi_m1,
    columns=X_common.columns,
).to_csv(OUT / "phi_m1_common_cohort.csv", index=False)

print("LDA phi:", phi_m1.shape)
print("LDA theta:", theta_m1.shape)


## 7. Exact submitted M7b architecture

The sparse model and warm-start procedure below follow the final M7b code in the supplied original notebook.

In [ ]:
class HGCTM(nn.Module):
    def __init__(
        self,
        K,
        F,
        L,
        A,
        prevalence_covariates=True,
        content_covariates=True,
        ordered_shrinkage=True,
        likelihood="dirichlet_multinomial",
        sigma_mu=2.0,
        sigma_litho=1.0,
        sigma_age=0.5,
        omega_a_init=0.5,
        kappa_init=50.0,
    ):
        super().__init__()
        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.prevalence_covariates = prevalence_covariates
        self.content_covariates = content_covariates
        self.ordered_shrinkage = ordered_shrinkage
        self.likelihood = likelihood
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age
        self.n_prev_covariates = L + A if prevalence_covariates else 0


class HGCTM_Sparse(HGCTM):
    def model(self, X, litho_idx, age_idx):
        N_local, F_local = X.shape
        K_local = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(K_local, F_local),
                self.sigma_mu * torch.ones(K_local, F_local),
            ).to_event(2),
        )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(K_local, self.L, F_local),
                self.sigma_litho
                * torch.ones(K_local, self.L, F_local),
            ).to_event(3),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(K_local, self.A, F_local),
                self.sigma_age
                * torch.ones(K_local, self.A, F_local),
            ).to_event(3),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        B = pyro.sample(
            "B",
            dist.Normal(
                torch.zeros(self.L + self.A, K_local - 1),
                torch.ones(self.L + self.A, K_local - 1),
            ).to_event(2),
        )

        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                torch.zeros(K_local - 1),
                torch.ones(K_local - 1),
            ).to_event(1),
        )

        log_kappa = pyro.sample(
            "log_kappa",
            dist.Normal(3.5, 1.0),
        )
        kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N_local):
            z_litho = torch.nn.functional.one_hot(
                litho_idx,
                self.L,
            ).float()
            z_age = torch.nn.functional.one_hot(
                age_idx,
                self.A,
            ).float()
            z = torch.cat([z_litho, z_age], dim=1)

            eta = z @ B + intercept
            eta_full = torch.cat(
                [eta, torch.zeros(N_local, 1)],
                dim=1,
            )
            theta = softmax(eta_full, dim=1)

            litho_dev = delta_litho[
                :, litho_idx, :
            ].permute(1, 0, 2)
            age_dev = delta_age[
                :, age_idx, :
            ].permute(1, 0, 2)

            gated_litho = (
                tau.unsqueeze(0).unsqueeze(0)
                * litho_dev
            )
            gated_age = (
                tau.unsqueeze(0).unsqueeze(0)
                * age_dev
            )

            psi = (
                mu.unsqueeze(0)
                + gated_litho
                + omega_a * gated_age
            )
            phi = softmax(psi, dim=2)

            p = torch.einsum("nk,nkf->nf", theta, phi)
            p = p.clamp(min=1e-8)
            p = p / p.sum(dim=1, keepdim=True)

            n_i = X.sum(dim=1)
            alpha = kappa * p

            pyro.sample(
                "obs",
                dist.DirichletMultinomial(
                    total_count=n_i.int(),
                    concentration=alpha,
                ),
                obs=X,
            )

    def guide(self, X, litho_idx, age_idx):
        _, F_local = X.shape
        K_local = self.K

        mu_loc = pyro.param(
            "mu_loc",
            torch.randn(K_local, F_local) * 0.1,
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5 * torch.ones(K_local, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "mu",
            dist.Normal(mu_loc, mu_scale).to_event(2),
        )

        dl_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(K_local, self.L, F_local),
        )
        dl_scale = pyro.param(
            "delta_litho_scale",
            0.2 * torch.ones(K_local, self.L, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(dl_loc, dl_scale).to_event(3),
        )

        da_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(K_local, self.A, F_local),
        )
        da_scale = pyro.param(
            "delta_age_scale",
            0.1 * torch.ones(K_local, self.A, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_age",
            dist.Normal(da_loc, da_scale).to_event(3),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "omega_a",
            dist.Beta(omega_a_a, omega_a_b),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "tau",
            dist.Beta(tau_a, tau_b).to_event(1),
        )

        B_loc = pyro.param(
            "B_loc",
            torch.zeros(self.L + self.A, K_local - 1),
        )
        B_scale = pyro.param(
            "B_scale",
            0.5 * torch.ones(self.L + self.A, K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "B",
            dist.Normal(B_loc, B_scale).to_event(2),
        )

        int_loc = pyro.param(
            "intercept_loc",
            torch.zeros(K_local - 1),
        )
        int_scale = pyro.param(
            "intercept_scale",
            0.5 * torch.ones(K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "intercept",
            dist.Normal(int_loc, int_scale).to_event(1),
        )

        lk_loc = pyro.param(
            "log_kappa_loc",
            torch.tensor(3.5),
        )
        lk_scale = pyro.param(
            "log_kappa_scale",
            torch.tensor(0.5),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "log_kappa",
            dist.Normal(lk_loc, lk_scale),
        )


## 8. Fitting, extraction, likelihood, and topic-alignment utilities

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


def make_warm_mu_init(phi_reference):
    phi_clipped = np.clip(phi_reference, 1e-6, None)
    mu_init = np.log(phi_clipped)
    mu_init = mu_init - mu_init.mean(axis=1, keepdims=True)
    return torch.tensor(mu_init, dtype=torch.float32)


def fit_m7b(
    model,
    X,
    litho_idx,
    age_idx,
    phi_m1_ref,
    seed,
    n_steps=5000,
    anneal_end=1500,
    lr=0.005,
    print_every=500,
):
    set_all_seeds(seed)
    pyro.clear_param_store()

    optimizer = ClippedAdam({
        "lr": lr,
        "betas": (0.9, 0.999),
    })
    svi = SVI(
        model.model,
        model.guide,
        optimizer,
        loss=Trace_ELBO(),
    )

    # Original notebook: one dummy step creates all parameters.
    svi.step(X, litho_idx, age_idx)

    pyro.param("mu_loc").data.copy_(
        make_warm_mu_init(phi_m1_ref)
    )

    F_local = X.shape[1]
    pyro.param("tau_a").data.copy_(
        50.0 * torch.ones(F_local)
    )
    pyro.param("tau_b").data.copy_(
        torch.ones(F_local)
    )

    losses = []

    for step in range(n_steps):
        loss = svi.step(X, litho_idx, age_idx)
        losses.append(float(loss))

        if step < anneal_end:
            progress = step / anneal_end
            min_tau_a = (
                50.0 * (1.0 - progress)
                + 1.0 * progress
            )
            with torch.no_grad():
                pyro.param("tau_a").data.clamp_(
                    min=min_tau_a
                )

        if (step + 1) % print_every == 0:
            tau_a = pyro.param("tau_a").detach()
            tau_b = pyro.param("tau_b").detach()
            tau_mean = (
                tau_a / (tau_a + tau_b)
            ).mean().item()
            phase = (
                "annealing"
                if step < anneal_end
                else "free"
            )
            print(
                f"    step {step+1:>5d}  "
                f"loss={loss:>12.1f}  "
                f"tau_mean={tau_mean:.3f}  "
                f"[{phase}]"
            )

    return losses


def extract_parameter_means():
    mu = pyro.param("mu_loc").detach().cpu().numpy()
    delta_litho = (
        pyro.param("delta_litho_loc")
        .detach().cpu().numpy()
    )
    delta_age = (
        pyro.param("delta_age_loc")
        .detach().cpu().numpy()
    )
    B = pyro.param("B_loc").detach().cpu().numpy()
    intercept = (
        pyro.param("intercept_loc")
        .detach().cpu().numpy()
    )

    tau_a = pyro.param("tau_a").detach().cpu().numpy()
    tau_b = pyro.param("tau_b").detach().cpu().numpy()
    tau = tau_a / (tau_a + tau_b)

    omega_a_a = float(pyro.param("omega_a_a").item())
    omega_a_b = float(pyro.param("omega_a_b").item())
    omega_a = omega_a_a / (omega_a_a + omega_a_b)

    kappa = float(
        np.exp(pyro.param("log_kappa_loc").item())
    )

    phi_global = np.exp(mu)
    phi_global = (
        phi_global
        / phi_global.sum(axis=1, keepdims=True)
    )

    return {
        "mu": mu,
        "phi_global": phi_global,
        "delta_litho": delta_litho,
        "delta_age": delta_age,
        "B": B,
        "intercept": intercept,
        "tau": tau,
        "omega_a": omega_a,
        "kappa": kappa,
    }


def deterministic_model_outputs(
    params,
    X,
    litho_idx,
    age_idx,
):
    mu = torch.tensor(params["mu"], dtype=torch.float32)
    delta_litho = torch.tensor(
        params["delta_litho"],
        dtype=torch.float32,
    )
    delta_age = torch.tensor(
        params["delta_age"],
        dtype=torch.float32,
    )
    B = torch.tensor(params["B"], dtype=torch.float32)
    intercept = torch.tensor(
        params["intercept"],
        dtype=torch.float32,
    )
    tau = torch.tensor(params["tau"], dtype=torch.float32)
    omega_a = torch.tensor(
        params["omega_a"],
        dtype=torch.float32,
    )

    N_local = X.shape[0]

    with torch.no_grad():
        z_litho = torch.nn.functional.one_hot(
            litho_idx,
            L,
        ).float()
        z_age = torch.nn.functional.one_hot(
            age_idx,
            A,
        ).float()
        z = torch.cat([z_litho, z_age], dim=1)

        eta = z @ B + intercept
        eta_full = torch.cat(
            [eta, torch.zeros(N_local, 1)],
            dim=1,
        )
        theta = softmax(eta_full, dim=1)

        litho_dev = delta_litho[
            :, litho_idx, :
        ].permute(1, 0, 2)
        age_dev = delta_age[
            :, age_idx, :
        ].permute(1, 0, 2)

        psi = (
            mu.unsqueeze(0)
            + tau.view(1, 1, -1) * litho_dev
            + omega_a
            * tau.view(1, 1, -1)
            * age_dev
        )
        phi_context = softmax(psi, dim=2)

        p = torch.einsum(
            "nk,nkf->nf",
            theta,
            phi_context,
        )
        p = p.clamp(min=1e-8)
        p = p / p.sum(dim=1, keepdim=True)

    return {
        "theta": theta.cpu().numpy(),
        "phi_context": phi_context.cpu().numpy(),
        "p": p.cpu().numpy(),
    }


def dirichlet_multinomial_log_prob(
    X,
    p,
    kappa,
):
    X_t = torch.tensor(X, dtype=torch.float64)
    p_t = torch.tensor(p, dtype=torch.float64)
    p_t = p_t.clamp(min=1e-12)
    p_t = p_t / p_t.sum(dim=1, keepdim=True)

    alpha = float(kappa) * p_t
    alpha0 = alpha.sum(dim=1)
    n_i = X_t.sum(dim=1)

    log_coefficient = (
        torch.lgamma(n_i + 1.0)
        - torch.lgamma(X_t + 1.0).sum(dim=1)
    )

    log_beta_ratio = (
        torch.lgamma(alpha0)
        - torch.lgamma(alpha0 + n_i)
        + (
            torch.lgamma(alpha + X_t)
            - torch.lgamma(alpha)
        ).sum(dim=1)
    )

    return (
        log_coefficient + log_beta_ratio
    ).cpu().numpy()


def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a) * np.linalg.norm(b)
    )
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)


def align_topics_to_reference(
    phi_run,
    phi_reference,
):
    similarity = np.zeros((K, K))

    for run_topic in range(K):
        for ref_topic in range(K):
            similarity[run_topic, ref_topic] = (
                cosine_similarity(
                    phi_run[run_topic],
                    phi_reference[ref_topic],
                )
            )

    run_indices, ref_indices = linear_sum_assignment(
        -similarity
    )

    ref_to_run = np.full(K, -1, dtype=int)
    run_to_ref = np.full(K, -1, dtype=int)

    for run_topic, ref_topic in zip(
        run_indices,
        ref_indices,
    ):
        ref_to_run[ref_topic] = run_topic
        run_to_ref[run_topic] = ref_topic

    aligned_phi = phi_run[ref_to_run]

    return {
        "aligned_phi": aligned_phi,
        "ref_to_run": ref_to_run,
        "run_to_ref": run_to_ref,
        "topic_cosines": np.array([
            similarity[ref_to_run[ref_topic], ref_topic]
            for ref_topic in range(K)
        ]),
        "similarity_matrix": similarity,
    }


def align_topic_axis(array, ref_to_run, axis=0):
    return np.take(array, ref_to_run, axis=axis)


def effect_strengths(params):
    tau = params["tau"]
    litho_eff = float(np.mean(
        np.abs(params["delta_litho"])
        * tau[None, None, :]
    ))
    age_eff = float(np.mean(
        np.abs(params["delta_age"])
        * tau[None, None, :]
        * params["omega_a"]
    ))

    return {
        "lithology_effective_mean_abs": litho_eff,
        "age_effective_mean_abs": age_eff,
        "ratio_lithology_over_age": (
            litho_eff / max(age_eff, 1e-12)
        ),
        "tau_mean": float(np.mean(tau)),
        "tau_median": float(np.median(tau)),
    }


## 9. Fit all three lithology-source models under identical settings

Three Pyro seeds are used by default. Every run starts from the same LDA topic matrix.

Results are saved immediately after each fit so that a later notebook interruption does not erase completed runs.

In [ ]:
run_summaries = []
run_phi_aligned = {}
primary_results = {}

for source_name in SOURCE_SPECS:
    lithology_tensor = source_lithology_tensors[source_name]

    run_phi_aligned[source_name] = {}

    for seed in ACTIVE_SEEDS:
        print("\n" + "=" * 76)
        print(f"SOURCE: {source_name} | SEED: {seed}")
        print("=" * 76)

        model = HGCTM_Sparse(
            K=K,
            F=F,
            L=L,
            A=A,
            prevalence_covariates=True,
            content_covariates=True,
            ordered_shrinkage=True,
            likelihood="dirichlet_multinomial",
            sigma_mu=2.0,
            sigma_litho=0.3,
            sigma_age=0.15,
        )

        start = time.time()

        losses = fit_m7b(
            model=model,
            X=X_tensor,
            litho_idx=lithology_tensor,
            age_idx=age_tensor,
            phi_m1_ref=phi_m1,
            seed=seed,
            n_steps=ACTIVE_STEPS,
            anneal_end=ACTIVE_ANNEAL_END,
            lr=LR,
            print_every=PRINT_EVERY,
        )

        elapsed = time.time() - start

        params = extract_parameter_means()
        outputs = deterministic_model_outputs(
            params,
            X_tensor,
            lithology_tensor,
            age_tensor,
        )
        log_prob_per_doc = dirichlet_multinomial_log_prob(
            X_np,
            outputs["p"],
            params["kappa"],
        )

        alignment = align_topics_to_reference(
            params["phi_global"],
            phi_m1,
        )

        ref_to_run = alignment["ref_to_run"]

        phi_aligned = alignment["aligned_phi"]
        delta_litho_aligned = align_topic_axis(
            params["delta_litho"],
            ref_to_run,
            axis=0,
        )
        delta_age_aligned = align_topic_axis(
            params["delta_age"],
            ref_to_run,
            axis=0,
        )
        theta_aligned = outputs["theta"][:, ref_to_run]

        strengths = effect_strengths(params)

        run_name = f"{source_name.lower()}_seed_{seed}"
        run_path = RUNS_DIR / f"{run_name}.npz"

        np.savez_compressed(
            run_path,
            source=np.array(source_name),
            seed=np.array(seed),
            Mindat_id=np.array(common_ids),
            family_names=np.array(X_common.columns),
            lithology_levels=np.array(LITHOLOGY_LEVELS),
            age_levels=np.array(AGE_LEVELS),
            losses=np.array(losses),
            phi_global_raw=params["phi_global"],
            phi_global_aligned=phi_aligned,
            mu_raw=params["mu"],
            delta_litho_raw=params["delta_litho"],
            delta_litho_aligned=delta_litho_aligned,
            delta_age_raw=params["delta_age"],
            delta_age_aligned=delta_age_aligned,
            B_raw=params["B"],
            intercept_raw=params["intercept"],
            tau=params["tau"],
            omega_a=np.array(params["omega_a"]),
            kappa=np.array(params["kappa"]),
            theta_raw=outputs["theta"],
            theta_aligned=theta_aligned,
            fitted_family_probabilities=outputs["p"],
            dm_log_prob_per_doc=log_prob_per_doc,
            ref_to_run=alignment["ref_to_run"],
            run_to_ref=alignment["run_to_ref"],
            topic_cosines_to_lda=alignment["topic_cosines"],
        )

        pd.DataFrame({
            "step": np.arange(1, len(losses) + 1),
            "loss": losses,
            "elbo": -np.array(losses),
        }).to_csv(
            RUNS_DIR / f"{run_name}_loss_trace.csv",
            index=False,
        )

        summary = {
            "source": source_name,
            "seed": seed,
            "n_deposits": N,
            "n_families": F,
            "n_steps": len(losses),
            "elapsed_seconds": elapsed,
            "elbo_last": -losses[-1],
            "elbo_mean_last_100": -float(
                np.mean(losses[-100:])
            ),
            "elbo_mean_last_100_per_doc": (
                -float(np.mean(losses[-100:])) / N
            ),
            "dm_log_prob_mean_per_doc": float(
                np.mean(log_prob_per_doc)
            ),
            "dm_log_prob_total": float(
                np.sum(log_prob_per_doc)
            ),
            "mean_topic_cosine_to_lda": float(
                alignment["topic_cosines"].mean()
            ),
            "min_topic_cosine_to_lda": float(
                alignment["topic_cosines"].min()
            ),
            "omega_a": params["omega_a"],
            "kappa": params["kappa"],
            **strengths,
        }
        run_summaries.append(summary)
        run_phi_aligned[source_name][seed] = phi_aligned

        if seed == PRIMARY_SEED:
            primary_results[source_name] = {
                "params": params,
                "outputs": outputs,
                "alignment": alignment,
                "phi_aligned": phi_aligned,
                "delta_litho_aligned": delta_litho_aligned,
                "delta_age_aligned": delta_age_aligned,
                "theta_aligned": theta_aligned,
                "summary": summary,
            }

        print(
            f"Finished {source_name}, seed {seed}: "
            f"ELBO/doc={summary['elbo_mean_last_100_per_doc']:.3f}, "
            f"DM log-prob/doc={summary['dm_log_prob_mean_per_doc']:.3f}, "
            f"topic cosine to LDA={summary['mean_topic_cosine_to_lda']:.3f}, "
            f"lithology/age={summary['ratio_lithology_over_age']:.2f}"
        )

run_summary_df = pd.DataFrame(run_summaries)
run_summary_df.to_csv(
    OUT / "full_data_run_summary.csv",
    index=False,
)

print("\nFull-data run summary:")
print(run_summary_df.round(4).to_string(index=False))


## 10. Across-seed stability and between-source topic comparison

In [ ]:
seed_stability_rows = []

for source_name in SOURCE_SPECS:
    reference_phi = run_phi_aligned[
        source_name
    ][PRIMARY_SEED]

    for seed in ACTIVE_SEEDS:
        phi_seed = run_phi_aligned[source_name][seed]
        topic_cosines = np.array([
            cosine_similarity(
                phi_seed[topic],
                reference_phi[topic],
            )
            for topic in range(K)
        ])

        for topic in range(K):
            seed_stability_rows.append({
                "source": source_name,
                "seed": seed,
                "reference_seed": PRIMARY_SEED,
                "topic_reference_order": topic + 1,
                "cosine_to_primary_seed": topic_cosines[topic],
            })

seed_stability = pd.DataFrame(seed_stability_rows)
seed_stability.to_csv(
    OUT / "within_source_seed_stability.csv",
    index=False,
)

source_pair_rows = []
source_names = list(SOURCE_SPECS)

for source_i in source_names:
    for source_j in source_names:
        phi_i = primary_results[source_i]["phi_aligned"]
        phi_j = primary_results[source_j]["phi_aligned"]

        topic_cosines = [
            cosine_similarity(phi_i[k], phi_j[k])
            for k in range(K)
        ]

        source_pair_rows.append({
            "source_1": source_i,
            "source_2": source_j,
            "mean_topic_cosine": float(np.mean(topic_cosines)),
            "minimum_topic_cosine": float(np.min(topic_cosines)),
            **{
                f"topic_{k+1}_cosine": topic_cosines[k]
                for k in range(K)
            },
        })

source_pairwise = pd.DataFrame(source_pair_rows)
source_pairwise.to_csv(
    OUT / "between_source_topic_alignment.csv",
    index=False,
)

print("Within-source seed stability:")
print(
    seed_stability.groupby("source")[
        "cosine_to_primary_seed"
    ].agg(["mean", "min", "std"]).round(4).to_string()
)

print("\nBetween-source primary-seed topic alignment:")
print(
    source_pairwise[
        [
            "source_1",
            "source_2",
            "mean_topic_cosine",
            "minimum_topic_cosine",
        ]
    ].round(4).to_string(index=False)
)


## 11. Topic tables and class-specific lithology-deviation magnitudes

In [ ]:
topic_rows = []
class_effect_rows = []
tau_rows = []

for source_name, result in primary_results.items():
    phi = result["phi_aligned"]
    delta_litho = result["delta_litho_aligned"]
    tau = result["params"]["tau"]

    for topic in range(K):
        top_indices = np.argsort(
            phi[topic]
        )[::-1][:10]

        for rank, family_index in enumerate(
            top_indices,
            start=1,
        ):
            topic_rows.append({
                "source": source_name,
                "seed": PRIMARY_SEED,
                "topic_reference_order": topic + 1,
                "rank": rank,
                "family": X_common.columns[family_index],
                "probability": phi[topic, family_index],
            })

    for lithology_index, lithology_class in enumerate(
        LITHOLOGY_LEVELS
    ):
        effective = (
            np.abs(delta_litho[:, lithology_index, :])
            * tau[None, :]
        )
        class_effect_rows.append({
            "source": source_name,
            "lithology_class": lithology_class,
            "mean_effective_absolute_deviation": float(
                effective.mean()
            ),
            "median_effective_absolute_deviation": float(
                np.median(effective)
            ),
        })

    for family, value in zip(
        X_common.columns,
        tau,
    ):
        tau_rows.append({
            "source": source_name,
            "family": family,
            "tau": float(value),
        })

topic_table = pd.DataFrame(topic_rows)
class_effect_table = pd.DataFrame(class_effect_rows)
tau_table = pd.DataFrame(tau_rows)

topic_table.to_csv(
    OUT / "primary_seed_top_families_by_source.csv",
    index=False,
)
class_effect_table.to_csv(
    OUT / "primary_seed_lithology_class_effect_magnitudes.csv",
    index=False,
)
tau_table.to_csv(
    OUT / "primary_seed_tau_by_source.csv",
    index=False,
)

print("Top families for the primary seed:")
for source_name in SOURCE_SPECS:
    print("\n", source_name)
    display = topic_table[
        (topic_table["source"] == source_name)
        & (topic_table["rank"] <= 5)
    ]
    print(
        display[
            [
                "topic_reference_order",
                "rank",
                "family",
                "probability",
            ]
        ].round(4).to_string(index=False)
    )


## 12. Optional common random holdout

This uses one identical train/test split for all three lithology versions. It is included only as a predictive sensitivity check.

It is **not** the corrected continent-holdout analysis requested for geographic transferability.

In [ ]:
holdout_summary_rows = []

if RUN_COMMON_HOLDOUT:
    stratification_labels = (
        cohort["Deposit_type"]
        .fillna("Unknown")
        .astype(str)
        .to_numpy()
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=HOLDOUT_FRACTION,
        random_state=HOLDOUT_SEED,
    )
    train_indices, test_indices = next(
        splitter.split(X_np, stratification_labels)
    )

    train_set = set(train_indices.tolist())
    test_set = set(test_indices.tolist())

    # Guarantee that every observed age or source-specific lithology
    # category has at least one training record.
    arrays_to_cover = {
        "age": age_idx_np,
        **source_lithology_idx,
    }

    moved_to_train = []

    for variable_name, values in arrays_to_cover.items():
        for category in np.unique(values):
            category_indices = np.where(
                values == category
            )[0]
            if not any(
                index in train_set
                for index in category_indices
            ):
                candidate = next(
                    index for index in category_indices
                    if index in test_set
                )
                test_set.remove(candidate)
                train_set.add(candidate)
                moved_to_train.append({
                    "variable": variable_name,
                    "category_index": int(category),
                    "row_index": int(candidate),
                    "Mindat_id": common_ids[candidate],
                })

    train_indices = np.array(
        sorted(train_set),
        dtype=int,
    )
    test_indices = np.array(
        sorted(test_set),
        dtype=int,
    )

    pd.DataFrame({
        "Mindat_id": common_ids,
        "split": [
            "train" if index in train_set else "test"
            for index in range(N)
        ],
    }).to_csv(
        OUT / "common_random_holdout_split.csv",
        index=False,
    )

    pd.DataFrame(moved_to_train).to_csv(
        OUT / "holdout_rows_moved_to_preserve_categories.csv",
        index=False,
    )

    X_train_np = X_np[train_indices]
    X_test_np = X_np[test_indices]

    X_train_t = torch.tensor(
        X_train_np,
        dtype=torch.float32,
    )
    X_test_t = torch.tensor(
        X_test_np,
        dtype=torch.float32,
    )
    age_train_t = age_tensor[train_indices]
    age_test_t = age_tensor[test_indices]

    lda_train = LatentDirichletAllocation(
        n_components=K,
        doc_topic_prior=0.5 / K,
        topic_word_prior=0.01,
        max_iter=300,
        random_state=42,
        learning_method="batch",
        n_jobs=-1,
    )
    lda_train.fit(X_train_np.astype(np.float64))
    phi_m1_train = (
        lda_train.components_
        / lda_train.components_.sum(axis=1, keepdims=True)
    )

    print(
        f"Common holdout: train={len(train_indices)}, "
        f"test={len(test_indices)}"
    )

    for source_name in SOURCE_SPECS:
        print("\n" + "-" * 70)
        print("Holdout source:", source_name)
        print("-" * 70)

        lithology_full_t = source_lithology_tensors[
            source_name
        ]
        lithology_train_t = lithology_full_t[
            train_indices
        ]
        lithology_test_t = lithology_full_t[
            test_indices
        ]

        model = HGCTM_Sparse(
            K=K,
            F=F,
            L=L,
            A=A,
            prevalence_covariates=True,
            content_covariates=True,
            ordered_shrinkage=True,
            likelihood="dirichlet_multinomial",
            sigma_mu=2.0,
            sigma_litho=0.3,
            sigma_age=0.15,
        )

        start = time.time()

        losses = fit_m7b(
            model=model,
            X=X_train_t,
            litho_idx=lithology_train_t,
            age_idx=age_train_t,
            phi_m1_ref=phi_m1_train,
            seed=HOLDOUT_MODEL_SEED,
            n_steps=ACTIVE_STEPS,
            anneal_end=ACTIVE_ANNEAL_END,
            lr=LR,
            print_every=PRINT_EVERY,
        )

        elapsed = time.time() - start
        params = extract_parameter_means()

        train_outputs = deterministic_model_outputs(
            params,
            X_train_t,
            lithology_train_t,
            age_train_t,
        )
        test_outputs = deterministic_model_outputs(
            params,
            X_test_t,
            lithology_test_t,
            age_test_t,
        )

        train_ll = dirichlet_multinomial_log_prob(
            X_train_np,
            train_outputs["p"],
            params["kappa"],
        )
        test_ll = dirichlet_multinomial_log_prob(
            X_test_np,
            test_outputs["p"],
            params["kappa"],
        )

        full_primary_phi = primary_results[
            source_name
        ]["params"]["phi_global"]
        alignment = align_topics_to_reference(
            params["phi_global"],
            full_primary_phi,
        )

        holdout_summary_rows.append({
            "source": source_name,
            "seed": HOLDOUT_MODEL_SEED,
            "n_train": len(train_indices),
            "n_test": len(test_indices),
            "elapsed_seconds": elapsed,
            "train_dm_log_prob_mean_per_doc": float(
                train_ll.mean()
            ),
            "test_dm_log_prob_mean_per_doc": float(
                test_ll.mean()
            ),
            "topic_alignment_to_full_data_mean": float(
                alignment["topic_cosines"].mean()
            ),
            "topic_alignment_to_full_data_min": float(
                alignment["topic_cosines"].min()
            ),
            "elbo_mean_last_100_per_train_doc": (
                -float(np.mean(losses[-100:]))
                / len(train_indices)
            ),
            "omega_a": params["omega_a"],
            "kappa": params["kappa"],
            **effect_strengths(params),
        })

    holdout_summary = pd.DataFrame(
        holdout_summary_rows
    )
    holdout_summary.to_csv(
        OUT / "common_random_holdout_summary.csv",
        index=False,
    )

    print("\nCommon random holdout summary:")
    print(
        holdout_summary.round(4).to_string(
            index=False
        )
    )
else:
    print("Common random holdout disabled.")


## 13. Summary figures

In [ ]:
# Figure 1: class distributions.
distribution_pivot = (
    source_distributions.pivot(
        index="lithology_class",
        columns="source",
        values="count",
    )
    .reindex(LITHOLOGY_LEVELS)
    .fillna(0)
)

ax = distribution_pivot.plot(
    kind="bar",
    figsize=(11, 5),
)
ax.set_xlabel("Lithology class")
ax.set_ylabel("Deposits")
ax.set_title(
    "Lithology assignments on the common 1,237-deposit cohort"
)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.savefig(
    FIG_DIR / "lithology_class_distributions.png",
    dpi=300,
)
plt.close()


# Figure 2: effect ratio by source and seed.
fig, ax = plt.subplots(figsize=(8, 5))
for source_name in SOURCE_SPECS:
    subset = run_summary_df[
        run_summary_df["source"] == source_name
    ]
    ax.scatter(
        [source_name] * len(subset),
        subset["ratio_lithology_over_age"],
        label=source_name,
    )
ax.set_ylabel("Effective lithology / age deviation ratio")
ax.set_title(
    "Lithology-to-age effect ratio across sources and seeds"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR / "lithology_age_ratio_by_source_seed.png",
    dpi=300,
)
plt.close()


# Figure 3: between-source primary-seed mean topic cosine.
pair_matrix = source_pairwise.pivot(
    index="source_1",
    columns="source_2",
    values="mean_topic_cosine",
).reindex(
    index=list(SOURCE_SPECS),
    columns=list(SOURCE_SPECS),
)

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(
    pair_matrix.values,
    vmin=0,
    vmax=1,
)
ax.set_xticks(range(len(pair_matrix.columns)))
ax.set_xticklabels(pair_matrix.columns)
ax.set_yticks(range(len(pair_matrix.index)))
ax.set_yticklabels(pair_matrix.index)
ax.set_title("Mean aligned topic cosine")

for row in range(pair_matrix.shape[0]):
    for column in range(pair_matrix.shape[1]):
        ax.text(
            column,
            row,
            f"{pair_matrix.iloc[row, column]:.3f}",
            ha="center",
            va="center",
        )

fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(
    FIG_DIR / "between_source_topic_cosine.png",
    dpi=300,
)
plt.close()

print("Saved figures:")
for path in sorted(FIG_DIR.iterdir()):
    print(" -", path.name)


## 14. Provenance, manifest, and downloadable archive

In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "HGCTM_Lithology_Source_Sensitivity_K7.ipynb",
    "original_notebook_sha256": "6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b",
    "input_directories": {
        "stage0": str(STAGE0),
        "phase1b": str(PHASE1B),
    },
    "cohort": {
        "original_model_n": int(X_model.shape[0]),
        "excluded_invalid_coordinate_n": int(
            len(excluded_coordinates)
        ),
        "common_valid_coordinate_n": int(N),
        "families": int(F),
        "common_ids_file": "common_cohort_covariates.csv",
    },
    "sources": SOURCE_SPECS,
    "fixed_levels": {
        "lithology": LITHOLOGY_LEVELS,
        "age": AGE_LEVELS,
    },
    "model": {
        "name": "M7b sparse HGCTM",
        "K": K,
        "sigma_mu": 2.0,
        "sigma_litho": 0.3,
        "sigma_age": 0.15,
        "likelihood": "Dirichlet-Multinomial",
        "tau_prior": "Beta(1,3)",
        "omega_age_prior": "Beta(2,2)",
        "steps": ACTIVE_STEPS,
        "anneal_end": ACTIVE_ANNEAL_END,
        "learning_rate": LR,
        "seeds": ACTIVE_SEEDS,
        "primary_seed": PRIMARY_SEED,
        "lda_warm_start_random_state": 42,
    },
    "common_random_holdout": {
        "run": bool(RUN_COMMON_HOLDOUT),
        "fraction_requested": HOLDOUT_FRACTION,
        "split_seed": HOLDOUT_SEED,
        "model_seed": HOLDOUT_MODEL_SEED,
        "note": (
            "Predictive source-sensitivity check only; "
            "not a continent-transfer analysis."
        ),
    },
    "quick_test": bool(QUICK_TEST),
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

environment = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "pyro": pyro.__version__,
}

with open(
    OUT / "environment_versions.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        environment,
        handle,
        indent=2,
    )

readme = '''
HGCTM lithology-source sensitivity at fixed K=7

This archive compares the final M7b sparse HGCTM using:
1. Macrostrat lithology
2. GLiM lithology
3. Macrostrat with transparent GLiM fallback

All models use the identical valid-coordinate cohort, mineral matrix,
age categories, category dimensions, LDA warm start, hyperparameters,
optimization settings, and seed list.

The 98 placeholder-coordinate records are excluded only from this
coordinate-derived lithology comparison.

This analysis tests sensitivity to lithology source. It does not replace
the separate equal-prior and learned-variance sensitivity analyses.
'''.strip()

(OUT / "README.txt").write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(WORK / "HGCTM_Lithology_Source_Sensitivity_K7_Outputs"),
    "zip",
    root_dir=OUT,
)

print("Output files:")
for path in sorted(OUT.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(OUT))

print("\nDownload from Kaggle Output:")
print(archive_path)
